# Fine-tune cpsam_dic v4 — brightness-augmented

Phase 3.5 retrain. Uses the brightness-augmented training set built
by `scripts/augment_brightness.py` (~21k pairs = 8× v3) so the model
sees ±60 brightness shifts, gamma 0.5/1.8, and vignette shading at
training time. Goal: ≥0.8 IoU retention on the brightness-perturbed
test set (verified after training via `scripts/bench_brightness.py`).

**Workflow before running this notebook:**
1. Locally: `python scripts/augment_brightness.py` (~5 min, CPU)
2. Locally: `python scripts/build_brightness_test.py` (~1 min)
3. Locally: zip `data/training/dic_splits_v4_brightness/` and upload
   to Drive at `cellscope_training/dic_splits_v4_brightness.zip`,
   then unzip in Drive UI.
4. Run this notebook on Colab (GPU runtime).
5. Locally: download the trained model to
   `cellscope/data/models/cpsam_dic_v4`.
6. Locally: `conda run -n cellpose4 python scripts/bench_brightness.py \
       --model data/models/cpsam_dic_v4 --label cpsam_dic_v4`
   then `python scripts/bench_brightness.py --compare \
       cpsam_dic_v2 cpsam_dic_v4`.

**Requirements**: GPU runtime (T4 free is fine, ~6 h for 20 epochs
on 1k subsample; A100 ~1.5 h for full 8k subsample).

**Output**: `cpsam_dic_v4` model file.

In [ ]:
# Step 1: install cellpose 4.x
!pip install 'cellpose>=4.1.1' tifffile -q

In [ ]:
# Step 1b (optional): keep Colab session alive
from IPython.display import display, Javascript
display(Javascript('''
function ConnectButton() {
    const b = document.querySelector("colab-connect-button");
    if (b) b.click();
}
setInterval(ConnectButton, 60000);
'''))
print('Keep-alive installed.')

In [ ]:
# Step 2: mount Drive and verify v4 brightness dataset
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/cellscope_training'
TRAIN_DIR = f'{DRIVE_PATH}/dic_splits_v4_brightness/train'

import os
if not os.path.exists(TRAIN_DIR):
    print('Brightness-augmented training data not found.')
    print('Expected:', TRAIN_DIR)
    print('See notebook header for the local prep steps.')
else:
    n = len(os.listdir(TRAIN_DIR))
    print(f'Found {n // 2} training pairs at {TRAIN_DIR}')

In [ ]:
# Step 3: stratified subsample so each brightness variant is represented
import glob, os, re, random
from collections import defaultdict
import tifffile

img_files = sorted(glob.glob(f'{TRAIN_DIR}/*_img.tif'))
# v4 filenames: <base>_<variant>_img.tif where variant is one of
# 'b30+', 'b60+', 'b30-', 'b60-', 'g05', 'g18', 'vig', or none (orig)
VARIANT_RE = re.compile(r'_(b30\+|b60\+|b30-|b60-|g05|g18|vig)_img\.tif$')
by_variant = defaultdict(list)
for f in img_files:
    m = VARIANT_RE.search(os.path.basename(f))
    by_variant[m.group(1) if m else 'orig'].append(f)
for k, v in sorted(by_variant.items()):
    print(f'  {k:<6} {len(v)} pairs')

# Subsample T4-friendly: 125 per variant × 8 = 1000 total.
# Bump per_variant or set to None on High-RAM / A100.
PER_VARIANT = 125          # set to None for full set
random.seed(0)
selected = []
for variant, files in by_variant.items():
    if PER_VARIANT and len(files) > PER_VARIANT:
        files = random.sample(files, PER_VARIANT)
    selected.extend(files)
random.shuffle(selected)

train_files, train_labels_files = [], []
for f in selected:
    mf = f.replace('_img.tif', '_masks.tif')
    if os.path.exists(mf):
        train_files.append(f)
        train_labels_files.append(mf)
print(f'\nFinal training set: {len(train_files)} pairs')
sample = tifffile.imread(train_files[0])
print(f'Sample shape: {sample.shape} dtype {sample.dtype}')

In [ ]:
# Step 4: fine-tune. Same heartbeat + per-epoch checkpoint pattern
# as the v3 notebook so a Colab disconnect doesn't lose progress.
import time, gc, threading
import cellpose
from cellpose import models, train

print(f'cellpose version: {cellpose.version}')
gc.collect()
try:
    import torch; torch.cuda.empty_cache()
except Exception:
    pass

model = models.CellposeModel(gpu=True)

N_EPOCHS = 20
LR = 1e-5
BATCH_SIZE = 1
SAVE_EVERY = 1

OUT_DIR = f'{DRIVE_PATH}/models'
os.makedirs(OUT_DIR, exist_ok=True)
MODEL_NAME = 'cpsam_dic_v4'

print(f'Training {N_EPOCHS} epochs, lr={LR}, batch={BATCH_SIZE}, '
      f'pairs={len(train_files)}, save_every={SAVE_EVERY}')
print(f'Output: {OUT_DIR}/{MODEL_NAME} (overwritten each epoch)')

def _heartbeat(stop):
    import psutil
    t0 = time.time()
    while not stop.is_set():
        mins = (time.time() - t0) / 60
        ram = psutil.virtual_memory()
        msg = (f'[heartbeat] +{mins:5.1f} min   '
               f'RAM {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB')
        try:
            import torch
            if torch.cuda.is_available():
                msg += f'   GPU {torch.cuda.memory_allocated()/1e9:.1f} GB'
        except Exception:
            pass
        print(msg, flush=True)
        for _ in range(60):
            if stop.is_set():
                return
            time.sleep(1)

stop = threading.Event()
hb = threading.Thread(target=_heartbeat, args=(stop,), daemon=True)
hb.start()

t0 = time.time()
try:
    new_path, train_losses, test_losses = train.train_seg(
        model.net,
        train_files=train_files,
        train_labels_files=train_labels_files,
        save_path=OUT_DIR,
        n_epochs=N_EPOCHS,
        learning_rate=LR,
        batch_size=BATCH_SIZE,
        save_every=SAVE_EVERY,
        model_name=MODEL_NAME,
        min_train_masks=1,
    )
finally:
    stop.set()
    hb.join(timeout=2)

print(f'\nDone in {(time.time()-t0)/60:.1f} minutes')
print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Model saved: {new_path}')

In [ ]:
# Step 5: quick validation — clean v3 val + brightness-perturbed val
import numpy as np
VAL_DIR = f'{DRIVE_PATH}/dic_splits_v3/val'
trained = models.CellposeModel(gpu=True, pretrained_model=new_path)

if os.path.exists(VAL_DIR):
    val_imgs = sorted(glob.glob(f'{VAL_DIR}/*_img.tif'))[:30]
    ious = []
    for f in val_imgs:
        img = tifffile.imread(f)
        gt = tifffile.imread(f.replace('_img.tif', '_masks.tif')) > 0
        pred, _, _ = trained.eval(img)
        pb = pred > 0
        inter = np.logical_and(pb, gt).sum()
        union = np.logical_or(pb, gt).sum()
        ious.append(inter / union if union > 0 else 0.0)
    print(f'Clean val IoU: {np.mean(ious):.3f} ({len(val_imgs)} frames)')

# Brightness sanity check: same images at +60 brightness
bright_dir = f'{DRIVE_PATH}/dic_splits_v3/test_brightness/b_plus_60'
if os.path.exists(bright_dir):
    test_imgs = sorted(glob.glob(f'{bright_dir}/*_img.tif'))[:30]
    ious = []
    for f in test_imgs:
        img = tifffile.imread(f)
        gt = tifffile.imread(f.replace('_img.tif', '_masks.tif')) > 0
        pred, _, _ = trained.eval(img)
        pb = pred > 0
        inter = np.logical_and(pb, gt).sum()
        union = np.logical_or(pb, gt).sum()
        ious.append(inter / union if union > 0 else 0.0)
    print(f'b_plus_60 IoU: {np.mean(ious):.3f} ({len(test_imgs)} frames)')
    print('(For full benchmark, download model and run scripts/bench_brightness.py locally.)')

In [ ]:
# Step 6: download trained model
from google.colab import files
files.download(new_path)
print('Place at: cellscope/data/models/cpsam_dic_v4')

## After downloading

Place the model file at `cellscope/data/models/cpsam_dic_v4`, then:

```bash
# Run the full brightness benchmark on the new model
conda run -n cellpose4 python scripts/bench_brightness.py \
    --model data/models/cpsam_dic_v4 --label cpsam_dic_v4

# Compare against the v2 baseline
python scripts/bench_brightness.py --compare cpsam_dic_v2 cpsam_dic_v4
```

Ship criterion: ≥0.8 retention on every perturbation in
`results/brightness_eval/comparison.md`. If met, promote v4 to the
default DIC model (update `data/models/cpsam_dic` symlink + the
modality config in `core/modality.py`).